# Averaging the coarse peaks of the 6 queries (iteration 27, offline)

**Question.** The coarse peak of the best query is 1.2-2.2 sigma off above 800 m, and the error is random. Each photo has 6 queries (headings -5/0/+5 x scales 1.0/1.05), each with its own peak. Does averaging the peaks that agree with the best-z peak bring the answer closer, with no matcher?

**Data.** `../results/coarse_confidence/queries_<set>.csv` (6 peaks per photo: x1, y1, z1, sigma) plus the truth from metadata and `store.json` (only for scoring). The truth is checked first: the error recomputed from x1,y1 must equal the stored err1.

**Methods.** `single` = peak of the query with the highest z (the baseline). `mean_r` / `median_r` / `zmean_r` = mean, median, z-weighted mean of the peaks within r sigma of the single peak (r = 1, 2). Score points per photo over all photos (wrong ones count -20).

**Kill rule (written before the run).** The averaging is useful only if, on all three sets, it does not add wrong answers (beyond 8 sigma) and raises the share within 1 sigma and the points per photo on the new set above 800 m.

In [1]:
import json, numpy as np, pandas as pd
from pyproj import Transformer
C = "../results/coarse_confidence/"; I = "../results/coarse_averaging/inputs/"
def pts(e): return np.where(e<=1,10,np.where(e<=2,6,np.where(e<=4,3,np.where(e<=8,0,-20))))
def load(tag):
    q = pd.read_csv(C+"queries_%s.csv"%tag); meta = pd.read_csv(I+"meta_%s.csv"%tag); info = json.load(open(I+"store_%s.json"%tag))
    to_map = Transformer.from_crs("EPSG:4326", info["crs"], always_xy=True)
    e, n = to_map.transform(meta.lon_gtruth.values, meta.lat_gtruth.values)
    truth = pd.DataFrame(dict(photo=np.arange(len(meta)), tx=e-info["east0"], ty=info["north0"]-n))
    q = q.merge(truth, on="photo")
    check = np.hypot(q.x1-q.tx, q.y1-q.ty)/q.sigma_m
    print(tag, "truth check: max |err1 - recomputed| =", float(np.abs(check-q.err1).max()))
    return q
sets = {"new (332)": load("highalt_dev332"), "official (500)": load("official500"), "NUDT (300)": load("nudt300")}

highalt_dev332 truth check: max |err1 - recomputed| = 2.2737367544323206e-13
official500 truth check: max |err1 - recomputed| = 2.2737367544323206e-13
nudt300 truth check: max |err1 - recomputed| = 1.1368683772161603e-13


In [2]:
def estimates(g):
    b = g.loc[g.z1.idxmax()]; out = {"single": (b.x1, b.y1)}
    d = np.hypot(g.x1-b.x1, g.y1-b.y1)/b.sigma_m
    for r in (1, 2):
        near = g[d <= r]
        out["mean_%d"%r] = (near.x1.mean(), near.y1.mean())
        out["median_%d"%r] = (near.x1.median(), near.y1.median())
        w = near.z1.values; out["zmean_%d"%r] = (np.average(near.x1, weights=w), np.average(near.y1, weights=w))
    return b, out
tables = {}
for name, q in sets.items():
    rows = []
    for p, g in q.groupby("photo"):
        b, est = estimates(g)
        row = dict(photo=p, height=b.height, votes=int((np.hypot(g.x1-b.x1, g.y1-b.y1)/b.sigma_m <= 1).sum()))
        for m, (x, y) in est.items():
            row[m] = np.hypot(x-b.tx, y-b.ty)/b.sigma_m
        rows.append(row)
    tables[name] = pd.DataFrame(rows)
methods = ["single", "mean_1", "median_1", "zmean_1", "mean_2", "median_2", "zmean_2"]
def summary(t, label):
    out = []
    for m in methods:
        e = t[m]; out.append(dict(method=m, n=len(t), within1=round((e<=1).mean(),3), within2=round((e<=2).mean(),3), wrong=int((e>8).sum()),
                                  median_not_wrong=round(e[e<=8].median(),2), pts_per_photo=round(pts(e.values).mean(),2)))
    print("\n==", label); print(pd.DataFrame(out).to_string(index=False))
for name, t in tables.items(): summary(t, name + ", all heights")
t = tables["new (332)"]; summary(t[t.height >= 800], "new, above 800 m")
for lo, hi in [(0,500),(500,800),(800,1000),(1000,1300),(1300,1600),(1600,2200)]:
    summary(t[(t.height>=lo)&(t.height<hi)], "new, %d-%d m" % (lo, hi))


== new (332), all heights
  method   n  within1  within2  wrong  median_not_wrong  pts_per_photo
  single 332    0.398    0.750     16              1.21           5.70
  mean_1 332    0.404    0.747     16              1.21           5.72
median_1 332    0.407    0.753     16              1.26           5.75
 zmean_1 332    0.401    0.747     16              1.20           5.71
  mean_2 332    0.380    0.753     16              1.16           5.65
median_2 332    0.389    0.759     16              1.20           5.70
 zmean_2 332    0.389    0.756     16              1.14           5.70

== official (500), all heights
  method   n  within1  within2  wrong  median_not_wrong  pts_per_photo
  single 500    0.742    0.896     52              0.55           6.26
  mean_1 500    0.744    0.896     52              0.54           6.27
median_1 500    0.738    0.896     52              0.55           6.25
 zmean_1 500    0.746    0.896     52              0.53           6.28
  mean_2 500    0.

In [3]:
# does it matter how many queries agree? new set above 800 m, by votes
h = tables["new (332)"]; h = h[h.height>=800]
print(h.groupby(h.votes.clip(upper=4)).agg(n=("single","size"), single_median=("single","median"), mean1_median=("mean_1","median")).round(2))

         n  single_median  mean1_median
votes                                  
1      102           2.08          2.08
2       32           1.37          1.37
3        5           1.21          1.21
4        3           1.40          1.40


## Why it does not help: is the error shared between queries? (oracle: only queries with err1 <= 3 sigma, new set above 800 m)
If the 6 peak errors were independent, the mean of k near-truth peaks would have about 1/sqrt(k) of the single error. If they are shared (same bias), it will not.

In [4]:
q = sets["new (332)"]; q = q[q.height >= 800]
single, mean_k, spread, ks = [], [], [], []
for p, g in q.groupby("photo"):
    near = g[g.err1 <= 3]
    if len(near) < 2: continue
    ex = near.x1 - near.tx; ey = near.y1 - near.ty; s = near.sigma_m.iloc[0]
    single.append(np.hypot(ex, ey).mean()/s); mean_k.append(np.hypot(ex.mean(), ey.mean())/s)
    spread.append(np.hypot(near.x1 - near.x1.mean(), near.y1 - near.y1.mean()).mean()/s); ks.append(len(near))
print("photos with >= 2 near-truth queries:", len(single), " mean k:", round(np.mean(ks),1))
print("mean error of one query: %.2f sigma | error of the mean of the near queries: %.2f sigma | spread of the queries around their own mean: %.2f sigma" % (np.mean(single), np.mean(mean_k), np.mean(spread)))
print("independent errors would give", round(np.mean(single)/np.sqrt(np.mean(ks)),2), "sigma for the mean")

photos with >= 2 near-truth queries: 79  mean k: 3.2
mean error of one query: 1.60 sigma | error of the mean of the near queries: 1.16 sigma | spread of the queries around their own mean: 1.04 sigma
independent errors would give 0.9 sigma for the mean
